# Top-P 采样（Nucleus Sampling）

> 动态选择最小 token 集使其累积概率 ≥ P，自适应截断。

## 背景
Top-K 的 K 是固定的，但不同位置的分布形状不同（有的集中有的分散）。
Top-P（nucleus sampling）动态选择累积概率恰好超过 P 的最小 token 集，
分布集中时选少量 token，分散时选更多，比 Top-K 更鲁棒。

## 公式
$$\text{TopP}(p, P) = \text{sample}\left(\frac{p \cdot \mathbb{1}[\sum_{j \leq i} p_{(j)} \leq P]}{\sum_{\text{selected}} p_i}\right)$$
其中 p_{(j)} 是按概率降序排列的。

## 复杂度
- 时间：O(V log V)，排序
- 空间：O(V)
- 典型 P：0.9-0.95

## 考察点
- P 的选择：P=0.9 是常用值，P=1.0 退化为纯采样
- 与 Top-K 对比：Top-P 自适应，分布集中时选更少 token
- 论文：Holtzman et al. 2020 "The Curious Case of Neural Text Degeneration"


In [ ]:
import torch
import torch.nn.functional as F

def topp_sampling(logits: torch.Tensor, p: float = 0.9, temperature: float = 1.0) -> torch.Tensor:
    """Nucleus (Top-P) 采样: 选累计概率刚超过 p 的最小 token 集合。"""
    logits = logits / temperature
    probs = F.softmax(logits, dim=-1)
    sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
    cumsum = torch.cumsum(sorted_probs, dim=-1)
    mask = cumsum - sorted_probs > p
    sorted_probs[mask] = 0.0
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
    sampled_idx = torch.multinomial(sorted_probs, 1)
    return sorted_indices[0, sampled_idx[0]]

# 验证: nucleus 概率和 >= p
logits = torch.randn(1, 100) * 2
p = 0.9
probs = F.softmax(logits, dim=-1)
sorted_probs, _ = torch.sort(probs, dim=-1, descending=True)
cumsum = torch.cumsum(sorted_probs, dim=-1)
nucleus_size = (cumsum <= p).sum().item() + 1
assert nucleus_size > 0, "nucleus should have at least 1 token"
assert cumsum[0, nucleus_size - 1] >= p - 0.01, f"nucleus cumsum too low: {cumsum[0, nucleus_size-1]}"

# 验证采样
sampled = topp_sampling(logits, p=0.9, temperature=1.0)
assert 0 <= sampled.item() < 100, "token out of vocab"
print(f"✅ TopP: p={p}, nucleus_size={nucleus_size}, sampled_token={sampled.item()}")


## 小结
- 截断条件易写反：要保留的是"累积**未**超过 p 的 + 第一个让累积超过 p 的"，等价于把 `cum - prob > p` 的置零。
- TopP 对长尾更鲁棒，是开放域生成（故事/对话）的默认选择；代码/事实类任务常用贪心或低温度。
- 工程上常 `temperature → topk → topp` 串联。

## ✅ 测试验证

In [ ]:
# 验证 Top-P (nucleus) 采样
import torch
import torch.nn.functional as F

logits = torch.randn(100)
P = 0.9

# Top-P: 按概率降序累加，直到累积概率 >= P，保留这些 token
probs = F.softmax(logits, dim=-1)
sorted_probs, sorted_indices = probs.sort(descending=True)
cumsum = sorted_probs.cumsum(dim=-1)

# 找到累积概率首次 >= P 的位置
cutoff = (cumsum <= P).sum().item() + 1  # +1 包含首次超过 P 的那个
nucleus = sorted_indices[:cutoff]

# 验证: 保留的 token 累积概率 >= P
nucleus_probs = probs[nucleus]
assert nucleus_probs.sum() >= P - 1e-6, f"nucleus sum {nucleus_probs.sum()} < P={P}"

# 验证: 去掉最小的那个就 < P
if cutoff > 1:
    nucleus_minus = nucleus[:-1]
    assert probs[nucleus_minus].sum() < P, "removing smallest should make sum < P"

print(f"✅ TopP 测试通过: nucleus 累积概率 {nucleus_probs.sum().item():.4f} >= P={P}")
